# Figure 1: Amortized neural OT rearrangement on the sinusoidal target

This notebook trains a neural OT map and a coverage-conditioned, measure-preserving rearrangement of that map. During rearrangement training, one coverage mass is sampled uniformly from $(0, 1)$ for every minibatch. The final figure compares the neural OT contour, the amortized rearranged contour at a selected coverage mass, and the oracle HDR contour.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import chi2

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent.parent

src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from configs.datasets.synthetic.sinusoidal_transport import SinusoidalTransportDatasetConfig
from configs.predictors.rearranged_transport import (
    AmortizedRearrangedTransportPredictorConfig,
)
from configs.predictors.transport import (
    NormalizingFlowPredictorConfig,
    NeuralOptimalTransportPredictorConfig
)
from configs.trainers.rearranged_transport import (
    AmortizedRearrangedTransportTrainerConfig
)
from configs.trainers.transport import (
    NormalizingFlowTrainerConfig,
    NeuralOptimalTransportTrainerConfig
)
from data.datasets.synthetic.sinusoidal_transport import SinusoidalTransportDataset
from data.loaders import make_xy_dataloader
from predictors.rearranged_transport import AmortizedRearrangedTransport
from predictors.transport import NormalizingFlowPredictor, NeuralOptimalTransportPredictor
from trainers.rearranged_transport import AmortizedRearrangedTransportTrainer
from trainers.transport import NormalizingFlowTrainer, NeuralOptimalTransportTrainer

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


In [ ]:
seed = 7
torch.manual_seed(seed)
np.random.seed(seed)

device = "cuda"
dtype = "float32"

n_train = 20_000
n_calibration = 1_024
n_test = 1_024
batch_size = 1_024

plotting_coverage_mass = 0.95
x_plot_value = 2.3

ot_epochs = 50
ot_warmup_iterations = 10
c_transform_max_iter = 1000

rearrangement_epochs = 100

n_boundary_points = 720
plot_batch_size = 256

output_path = Path("../figures/figure_1_neural_ot_amortized_rearranged.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
dataset_config = SinusoidalTransportDatasetConfig(
    n_train=n_train,
    n_calibration=n_calibration,
    n_test=n_test,
    x_dim=1,
    y_dim=2,
    seed=seed,
    device=device,
    dtype=dtype,
)

dataset = SinusoidalTransportDataset(dataset_config)
splits = dataset.get_splits()

train_dataloader = make_xy_dataloader(
    data=splits.train,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

print("train x/y:", splits.train.x.shape, splits.train.y.shape)
print("calibration x/y:", splits.calibration.x.shape, splits.calibration.y.shape)
print("test x/y:", splits.test.x.shape, splits.test.y.shape)

In [ ]:
predictor_config = NeuralOptimalTransportPredictorConfig(
    x_dim=dataset.x_dim,
    y_dim=dataset.y_dim,
    hidden_dim=64,
    num_hidden_layers=8,
    c_transform_lr=1.,
    c_transform_max_iter=10000,
    seed=seed,
    device=device,
    dtype=dtype,
)

trainer_config = NeuralOptimalTransportTrainerConfig(
    epochs=ot_epochs,
    learning_rate=1e-3,
    weight_decay=1e-4,
    warmup_iterations=ot_warmup_iterations,
    grad_clip_norm=1.0,
    use_cosine_scheduler=True,
    verbose=True,
)

predictor = NeuralOptimalTransportPredictor(predictor_config)
trainer = NeuralOptimalTransportTrainer(trainer_config)

predictor = predictor.load("../../weights/neural_ot_predictor.pt")

# predictor = trainer.fit(
#     predictor=predictor,
#     dataloader=train_dataloader,
# )
predictor.eval()

trainer.training_history[-5:]

In [ ]:
# predictor.save("../../weights/neural_ot_predictor.pt")

In [ ]:
amortized_predictor_config = AmortizedRearrangedTransportPredictorConfig(
    x_dim=dataset.x_dim,
    y_dim=dataset.y_dim,
    hidden_dimension=24,
    number_of_hidden_layers=4,
    use_adjoint=True,
    method="euler",
    vector_field_implementation="sparse",
    activation="silu",
    number_of_steps=8,
    seed=seed,
    device=device,
    dtype=dtype,
)

amortized_predictor = AmortizedRearrangedTransport(
    config=amortized_predictor_config,
    transport_predictor=predictor,
)

amortized_trainer_config = AmortizedRearrangedTransportTrainerConfig(
    epochs=rearrangement_epochs,
    train_transport_map=False,
    learning_rate=1e-3,
    weight_decay=1e-4,
    mc_samples_per_x=64,
    grad_clip_norm=1.0,
    use_cosine_scheduler=True,
    verbose=True,
)

amortized_trainer = AmortizedRearrangedTransportTrainer(
    amortized_trainer_config
)
amortized_predictor = amortized_trainer.fit(
    predictor=amortized_predictor,
    dataloader=train_dataloader,
)
amortized_predictor.eval()

amortized_trainer.training_history[-5:]


In [ ]:
plt.plot([epoch['log_volume_loss'] for epoch in amortized_trainer.training_history])

In [ ]:
def circle_boundary(radius, n_points):
    theta = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=False)
    boundary = np.column_stack([np.cos(theta), np.sin(theta)]) * radius
    return torch.as_tensor(boundary, device=dataset.device, dtype=dataset.dtype)


def close_contour(contour):
    contour = np.asarray(contour)
    if np.allclose(contour[0], contour[-1]):
        return contour
    return np.vstack([contour, contour[0]])


def repeated_x(batch_size, x_value=x_plot_value):
    return torch.full(
        (batch_size, dataset.x_dim),
        fill_value=x_value,
        device=dataset.device,
        dtype=dataset.dtype,
    )


def model_pushforward_contour(
    model,
    u,
    alpha=None,
    batch_size=plot_batch_size,
):
    mapped = []
    for start in range(0, u.shape[0], batch_size):
        u_batch = u[start:start + batch_size]
        x_batch = repeated_x(u_batch.shape[0]).to(u_batch)

        if alpha is None:
            y_batch = model.pushforward(x=x_batch, u=u_batch)
        else:
            y_batch = model.pushforward(
                x=x_batch,
                u=u_batch,
                alpha=alpha,
            )

        mapped.append(y_batch.detach().cpu())

    return close_contour(torch.cat(mapped, dim=0).numpy())


def oracle_pushforward_contour(u, batch_size=plot_batch_size):
    mapped = []
    for start in range(0, u.shape[0], batch_size):
        u_batch = u[start:start + batch_size]
        x_batch = repeated_x(u_batch.shape[0]).to(u_batch)
        y_batch = dataset.push_u_given_x(u=u_batch, x=x_batch)
        mapped.append(y_batch.detach().cpu())
    return close_contour(torch.cat(mapped, dim=0).numpy())


def contour_bounds(contours, side_margin=0.05, bottom_margin=0.10, top_margin=0.20):
    points = np.vstack([np.asarray(contour) for contour in contours])
    xmin, ymin = points.min(axis=0)
    xmax, ymax = points.max(axis=0)
    width = max(xmax - xmin, 1e-12)
    height = max(ymax - ymin, 1e-12)
    return (
        xmin - side_margin * width,
        xmax + side_margin * width,
        ymin - bottom_margin * height,
        ymax + top_margin * height,
    )


In [ ]:
r_source = float(
    np.sqrt(chi2.ppf(plotting_coverage_mass, df=dataset.y_dim))
)
source_ball_boundary = circle_boundary(
    radius=r_source,
    n_points=n_boundary_points,
)

neural_ot_contour = model_pushforward_contour(
    predictor,
    source_ball_boundary,
)
amortized_contour = model_pushforward_contour(
    amortized_predictor,
    source_ball_boundary,
    alpha=plotting_coverage_mass,
)
oracle_hdr_contour = oracle_pushforward_contour(source_ball_boundary)

xmin, xmax, ymin, ymax = contour_bounds([
    neural_ot_contour,
    amortized_contour,
    oracle_hdr_contour,
])

print(f"coverage_mass = {plotting_coverage_mass:.3f}")
print(f"x = {x_plot_value:.3f}")
print(f"source radius = {r_source:.4f}")

fig, ax = plt.subplots(figsize=(5.8, 4.8), constrained_layout=True)

ax.plot(
    neural_ot_contour[:, 0],
    neural_ot_contour[:, 1],
    color="forestgreen",
    linewidth=2.0,
    label="Neural OT",
)
ax.plot(
    amortized_contour[:, 0],
    amortized_contour[:, 1],
    color="royalblue",
    linewidth=2.0,
    label="Amortized rearranged Neural OT",
)
ax.plot(
    oracle_hdr_contour[:, 0],
    oracle_hdr_contour[:, 1],
    color="darkorange",
    linewidth=2.0,
    label="Oracle HDR",
)

ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel(r"$y_1$")
ax.set_ylabel(r"$y_2$")
ax.set_title(
    rf"Pushforward contours at $x={x_plot_value:.1f}$, "
    rf"mass={plotting_coverage_mass:.2f}"
)
ax.legend(frameon=False, loc="upper right")

fig.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.02,
    metadata={"Creator": "Matplotlib"},
)

plt.show()


In [ ]:
u = torch.randn(
    1024,
    dataset.y_dim,
    device=dataset.device,
    dtype=dataset.dtype,
)
x = repeated_x(u.shape[0]).to(u)
u_np = u.detach().cpu().numpy()
u_pf_np = amortized_predictor.rearrangement_pushforward(
    x=x,
    u=u,
    alpha=plotting_coverage_mass,
).detach().cpu().numpy()


x_boundary = repeated_x(source_ball_boundary.shape[0]).to(
    source_ball_boundary
)

scale_to_pf_boundary = {}
scale_to_sc_boundary = {}
for scale in [1., 2., 3., 4., 5., 6., 7., 8.]:
    scale /= 8.
    scaled_source_boundary = source_ball_boundary * scale
    scale_to_sc_boundary[scale] = (
        scaled_source_boundary.detach().cpu().numpy()
    )
    scale_to_pf_boundary[scale] = (
        amortized_predictor.rearrangement_pushforward(
            x=x_boundary,
            u=scaled_source_boundary,
            alpha=plotting_coverage_mass,
        ).detach().cpu().numpy()
    )


In [ ]:
plt.scatter(u_np[:, 0], u_np[:, 1])
plt.scatter(u_pf_np[:, 0], u_pf_np[:, 1])
plt.show()

for scale in scale_to_sc_boundary.keys():

    plt.plot(scale_to_sc_boundary[scale][:, 0], scale_to_sc_boundary[scale][:, 1], c="red")
    plt.plot(scale_to_pf_boundary[scale][:, 0], scale_to_pf_boundary[scale][:, 1], c='green')
    
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import Normalize

_velocity_grid_resolution = 17
_velocity_plot_limit = 1.3 * r_source
_velocity_plot_axis = np.linspace(
    -_velocity_plot_limit,
    _velocity_plot_limit,
    _velocity_grid_resolution,
)
_velocity_u_1, _velocity_u_2 = np.meshgrid(
    _velocity_plot_axis,
    _velocity_plot_axis,
    indexing="xy",
)
_velocity_plot_points = np.column_stack([
    _velocity_u_1.ravel(),
    _velocity_u_2.ravel(),
])
_velocity_plot_points_tensor = torch.as_tensor(
    _velocity_plot_points,
    device=amortized_predictor.device,
    dtype=amortized_predictor.dtype,
)
_velocity_source_boundary = close_contour(
    source_ball_boundary.detach().cpu().numpy()
)


def _evaluate_velocity_field(time, batch_size=256):
    velocity_batches = []
    vector_field = amortized_predictor.rearrangement_flow.vector_field

    for start in range(0, _velocity_plot_points_tensor.shape[0], batch_size):
        states = (
            _velocity_plot_points_tensor[start:start + batch_size]
            .detach()
            .clone()
            .requires_grad_(True)
        )
        covariates = states.new_full(
            (states.shape[0], amortized_predictor.x_dim),
            float(x_plot_value),
        )
        coverage = states.new_full(
            (states.shape[0], 1),
            float(plotting_coverage_mass),
        )
        context = torch.cat([covariates, coverage], dim=-1)

        with torch.enable_grad():
            velocity = vector_field(u=states, x=context, t=float(time))

        velocity_batches.append(velocity.detach().cpu())

    return (
        torch.cat(velocity_batches, dim=0)
        .numpy()
        .reshape(_velocity_grid_resolution, _velocity_grid_resolution, 2)
    )


_velocity_time_selector = widgets.FloatsInput(
    value=[0.0, 0.25, 0.5, 0.75, 1.0],
    min=0.0,
    max=1.0,
    step=0.05,
    format=".2f",
    allow_duplicates=False,
    description="Timesteps:",
    layout=widgets.Layout(width="600px"),
    style={"description_width": "initial"},
)
_velocity_plot_button = widgets.Button(
    description="Plot velocity fields",
    icon="refresh",
)
_velocity_plot_output = widgets.Output()
_velocity_figure = None


def _plot_selected_velocity_fields(_=None):
    global _velocity_figure

    selected_times = tuple(sorted({
        float(time) for time in _velocity_time_selector.value
    }))

    with _velocity_plot_output:
        _velocity_plot_output.clear_output(wait=True)

        if _velocity_figure is not None:
            plt.close(_velocity_figure)
            _velocity_figure = None

        if not selected_times:
            print("Select at least one timestep between 0 and 1.")
            return

        if any(time < 0.0 or time > 1.0 for time in selected_times):
            print("All selected timesteps must be between 0 and 1.")
            return

        amortized_predictor.eval()
        velocity_fields = [
            _evaluate_velocity_field(time) for time in selected_times
        ]
        speeds = [
            np.linalg.norm(velocity, axis=-1)
            for velocity in velocity_fields
        ]
        maximum_speed = max(float(speed.max()) for speed in speeds)
        display_maximum_speed = max(maximum_speed, np.finfo(float).eps)
        color_normalization = Normalize(
            vmin=0.0,
            vmax=display_maximum_speed,
        )
        grid_spacing = _velocity_plot_axis[1] - _velocity_plot_axis[0]
        quiver_scale = display_maximum_speed / (0.75 * grid_spacing)

        number_of_columns = min(3, len(selected_times))
        number_of_rows = int(np.ceil(len(selected_times) / number_of_columns))
        _velocity_figure, axes = plt.subplots(
            number_of_rows,
            number_of_columns,
            figsize=(4.0 * number_of_columns, 3.7 * number_of_rows),
            sharex=True,
            sharey=True,
            squeeze=False,
            constrained_layout=True,
        )
        visible_axes = []
        quiver = None

        for axis, time, velocity, speed in zip(
            axes.flat, selected_times, velocity_fields, speeds
        ):
            quiver = axis.quiver(
                _velocity_u_1,
                _velocity_u_2,
                velocity[..., 0],
                velocity[..., 1],
                speed,
                cmap="viridis",
                norm=color_normalization,
                angles="xy",
                scale_units="xy",
                scale=quiver_scale,
                pivot="mid",
                width=0.004,
            )
            axis.plot(
                _velocity_source_boundary[:, 0],
                _velocity_source_boundary[:, 1],
                color="0.35",
                linestyle="--",
                linewidth=1.0,
                alpha=0.8,
            )
            axis.set(
                xlim=(-_velocity_plot_limit, _velocity_plot_limit),
                ylim=(-_velocity_plot_limit, _velocity_plot_limit),
                aspect="equal",
                xlabel=r"$u_1$",
                ylabel=r"$u_2$",
                title=rf"$t={time:.2f}$",
            )
            axis.grid(alpha=0.15)
            visible_axes.append(axis)

        for axis in axes.flat[len(selected_times):]:
            _velocity_figure.delaxes(axis)

        colorbar = _velocity_figure.colorbar(
            quiver,
            ax=visible_axes,
            shrink=0.82,
            pad=0.02,
        )
        _velocity_figure.suptitle(
            "Learned amortized rearrangement velocity field "
            rf"at $x={x_plot_value:.2f}$, mass={plotting_coverage_mass:.2f}"
        )
        plt.show()


_velocity_plot_button.on_click(_plot_selected_velocity_fields)
display(widgets.VBox([
    widgets.HTML(
        "Select one or more timesteps in [0, 1], then refresh the plot."
    ),
    _velocity_time_selector,
    _velocity_plot_button,
    _velocity_plot_output,
]))
_plot_selected_velocity_fields()
